In [39]:
import pandas as pd
import numpy as np
from collections import Counter
import ast

In [17]:
train_df = pd.read_csv("../data/train_translated_tokenized.csv")
train_df.rename(columns={train_df.columns[0]: "real_dataset_index"}, inplace=True)
val_df = pd.read_csv("../data/validation_translated.csv")
val_df.rename(columns={val_df.columns[0]: "real_dataset_index"}, inplace=True)

In [38]:
print(train_df["question_stripped"][0])

['30년', '전쟁의', '승자는', '누구인가']


In [ ]:
def generate_n_gram_table(df: pd.DataFrame, lang: str, n: int = 2, normalize: bool = True) -> pd.DataFrame:
    lang_df = df[df["lang"] == lang].copy()
    
    lang_df["question_stripped"] = lang_df["question_stripped"].apply(ast.literal_eval)
    
    n_gram_counter = Counter()

    for tokens in lang_df["question_stripped"]:
        tokens = ["<START>"] * (n - 1) + tokens + ["<END>"] * (n - 1)
        n_gram_counter.update(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))
    
    df_matrix = pd.DataFrame(
        [(ngram[:-1], ngram[-1], count) for ngram, count in n_gram_counter.items()],
        columns=["prefix", "next_word", "count"]
    )
    df_matrix["prefix"] = df_matrix["prefix"].apply(lambda x: " ".join(x))
    
    pivot_table = df_matrix.pivot_table(
        index="prefix", 
        columns="next_word", 
        values="count", 
        fill_value=0
    )
    
    if normalize:
        pivot_table = pivot_table.div(pivot_table.sum(axis=1), axis=0)
    
    return pivot_table

In [59]:
print(generate_n_gram_table(train_df, lang="ko", n=2).head)

<bound method NDFrame.head of next_word       100대  100위권에  10배에서  10세가  12호는  14세  15세의  15호  1610년  ...  \
prefix                                                                  ...   
           0.0   0.0     0.0    0.0   0.0   0.0  0.0   0.0  0.0    0.0  ...   
100대       0.0   0.0     0.0    0.0   0.0   0.0  0.0   0.0  0.0    0.0  ...   
100위권에     0.0   0.0     0.0    0.0   0.0   0.0  0.0   0.0  0.0    0.0  ...   
10배에서      0.0   0.0     0.0    0.0   0.0   0.0  0.0   0.0  0.0    0.0  ...   
10세가       0.0   0.0     0.0    0.0   0.0   0.0  0.0   0.0  0.0    0.0  ...   
...        ...   ...     ...    ...   ...   ...  ...   ...  ...    ...  ...   
히틀러의       0.0   0.0     0.0    0.0   0.0   0.0  0.0   0.0  0.0    0.0  ...   
힌국에서       0.0   0.0     0.0    0.0   0.0   0.0  0.0   0.0  0.0    0.0  ...   
힌두교의       0.0   0.0     0.0    0.0   0.0   0.0  0.0   0.0  0.0    0.0  ...   
힐은         0.0   0.0     0.0    0.0   0.0   0.0  0.0   0.0  0.0    0.0  ...   
힘러는        0.0   0.0  